# Lab 06 — AR modelling: fit, order, and residual checks

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 6 — §6.6–§6.8 (AR/MA/ARMA, order selection via ACF/PACF, an AR spectrum with a real peak), §6.3 (autocorrelation).

**Biomedical question.** Does my AR model actually fit — or is it hallucinating a peak?
**Task type (§1.8).** Stochastic (parametric) modelling — claim discipline for a spectral estimate.
**Information that must be preserved.** an honest link between the model and the data — a fit you can defend (right order, white residuals), not just a pretty spectrum.
**Main assumptions.** the signal is a stationary AR process driven by white noise, so its spectrum is fully set by a few pole locations.
**Primary diagnostic.** order selection by an information criterion (MDL/BIC) plus a residual-whiteness (ACF) check at the chosen order.
**Transfer challenge.** does the selected order — and the whiteness verdict — survive on a new recording, or does the 'peak' move / the residuals colour?

*Self-contained: a seeded synthetic AR(2) process (`np.random.default_rng(2013)`), numpy + scipy only, no `bsp`, no I/O. Yule-Walker is implemented by hand. Runs top-to-bottom in well under a minute.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab06_ar_fit_residual_checks/lab06_ar_fit_residual_checks.ipynb) [![nbviewer](https://img.shields.io/badge/view-nbviewer-orange)](https://nbviewer.org/github/farhad-abtahi/CM2013/blob/main/labs/lab06_ar_fit_residual_checks/lab06_ar_fit_residual_checks.ipynb) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab06_ar_fit_residual_checks.ipynb)

In [ ]:
# --- shared setup (reproducible; self-contained; no data files, no bsp) ---
import numpy as np, matplotlib.pyplot as plt
from scipy.signal import lfilter, find_peaks
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

# A KNOWN AR(2) process: a complex-conjugate pole pair at radius rho and angle 2*pi*f0 gives ONE
# sharp resonance at f0 (cycles/sample, i.e. f0*fs), driven by white noise. With the convention
#   x[n] = a1 x[n-1] + a2 x[n-2] + e[n]   <=>   A(z) = 1 - a1 z^-1 - a2 z^-2 = (1-rho e^{+jt})(1-rho e^{-jt})
#   =>  a1 = 2 rho cos(2 pi f0),   a2 = -rho^2         (t = 2 pi f0)
fs   = 1.0            # normalized frequency (cycles/sample); the resonance sits at f0*fs
f0   = 0.15          # true resonance frequency (~0.15 fs)
rho  = 0.95          # pole radius -> sharpness of the resonance (closer to 1 = sharper)
a1_true, a2_true = 2*rho*np.cos(2*np.pi*f0), -rho**2
sigma2_true = 1.0    # white driving-noise variance
N, burn = 600, 500
e = np.sqrt(sigma2_true) * rng.standard_normal(N + burn)
x = lfilter([1.0], [1.0, -a1_true, -a2_true], e)[burn:]   # AR(2); drop the start-up transient
print(f"true AR(2): a1={a1_true:+.4f}, a2={a2_true:+.4f}, resonance f0={f0} cyc/sample, N={N}")


## The estimator — Yule-Walker AR by hand

An AR(p) model says each sample is a linear echo of its own past plus white noise: `x[n] = a_1 x[n-1] + … + a_p x[n-p] + e[n]`. The **Yule-Walker** (autocorrelation) method fits it from the signal's autocorrelation `r[k]`: solve the Toeplitz **normal equations** `R a = [r[1]…r[p]]ᵀ` for the coefficients, and read the driving-noise variance off the prediction error `σ² = r[0] − Σ a_k r[k]`. The model spectrum is then `S(f) = σ² / |1 − Σ_k a_k e^{−j2πfk}|²` (one-sided). We build all of this with numpy — no statsmodels.

In [ ]:
# The estimator, BY HAND (no statsmodels): Yule-Walker = the autocorrelation method.
def yule_walker(x, p):
    """Fit AR(p) by Yule-Walker. Returns (a, sigma2) for x[n]=sum_k a_k x[n-k]+e[n],
    with a=[a_1..a_p] and sigma2 the driving-noise (prediction-error) variance."""
    # TODO: mean-remove x; biased autocorrelation r[0..p]; build the p x p symmetric Toeplitz
    #   matrix R (R[i,j]=r[|i-j|]); solve R a = [r[1]..r[p]] for a; sigma2 = r[0] - sum_k a_k r[k].
    raise NotImplementedError("TODO: implement this — see the comment above")

def ar_psd(a, sigma2, n_freq=1024):
    """One-sided AR PSD on f in [0,0.5] cyc/sample: S(f)=sigma2/|1 - sum_k a_k e^{-j2pi f k}|^2."""
    raise NotImplementedError("TODO: implement this — see the comment above")

def ar_residuals(x, a):
    """One-step-ahead prediction residuals e[n]=x[n]-sum_k a_k x[n-k], n>=p (the AR whitening filter)."""
    raise NotImplementedError("TODO: implement this — see the comment above")

def acf(z, maxlag):
    """Biased normalized autocorrelation r[k]/r[0], k=0..maxlag."""
    raise NotImplementedError("TODO: implement this — see the comment above")

# quick check: does YW recover the KNOWN AR(2) coefficients? (numbers computed live)
a_hat, s2_hat = yule_walker(x, 2)
print(f"YW p=2 estimate: a1={a_hat[0]:+.4f} (true {a1_true:+.4f}), "
      f"a2={a_hat[1]:+.4f} (true {a2_true:+.4f}), sigma2={s2_hat:.3f} (true {sigma2_true})")


## 1. Order controls the story — under-fit, matched, over-fit

Fit AR models of growing order and overlay their PSDs against the *known* truth. Too low an order is **too smooth** and misses the resonance; the matched order recovers it; too high an order starts **inventing spurious peaks** by fitting the noise. Print the estimated peak frequency at each order and watch where it drifts.

In [ ]:
# TODO fit AR(p) for p in the sweep via yule_walker, overlay their AR PSDs (semilogy), and print
#   the estimated peak frequency + number of spectral peaks at each order. The arc to show:
#   under-fit (AR(1), monotone, no resonance) -> matched (AR(2)) -> over-fit (AR(30), spurious peaks).
orders = [1, 2, 8, 30]          # AR(1) = the under-fit anchor; 2/8/30 = the requested sweep
f_true, S_true = ar_psd([a1_true, a2_true], sigma2_true)
plt.figure(figsize=(9, 4))
plt.semilogy(f_true, S_true, "k--", lw=2, label="true AR(2) PSD")
raise NotImplementedError("TODO: implement this — see the comment above")
plt.axvline(f0, color="grey", ls=":", label=f"true f0={f0}")
plt.xlabel("normalized frequency (cycles/sample)"); plt.ylabel("PSD")
plt.title("AR PSD vs model order"); plt.legend(fontsize=8); plt.show()
# Checkpoint: AR(1) is monotone (peak at 0 -> misses the resonance); AR(2)/AR(8) nail f0.
#   How many EXTRA peaks does AR(30) invent, and would you report any of them as a finding?


## 2. Pick the order honestly — an information criterion

You can't eyeball the order in practice. An information criterion trades fit (`N ln σ²(p)`, which only falls as p grows) against a complexity penalty. **MDL/BIC** (`p ln N`) penalises harder than **AIC** (`2p`) and is *consistent*, so we select on it. The minimum should land at — or just above — the true order 2.

In [ ]:
# TODO order selection: for p=1..pmax fit YW, take the prediction-error variance sigma2(p), and
#   compute an information criterion. Use MDL/BIC = N*ln(sigma2) + p*ln(N) for the SELECTION (consistent),
#   and AIC = N*ln(sigma2) + 2p alongside for contrast. Report the BIC-minimum order; confirm ~ true 2.
pmax = 30
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: how sharply does the criterion drop at p=2 vs the slow climb of the penalty after?
#   Does the data-driven order match the physics (a single pole pair needs order 2)?


## 3. The check that keeps you honest — residual whiteness

A good AR fit **whitens** the signal: the one-step-prediction residuals should look like white noise, so their autocorrelation should sit inside the ±1.96/√M band at every nonzero lag. If a lag pokes out, the model is still missing structure — and its pretty spectrum is not yet trustworthy. This residual check, not the shape of the PSD, is what lets you *defend* the fit.

In [ ]:
# TODO residual whiteness at the SELECTED order: compute one-step-prediction residuals with
#   yule_walker(x, sel_order) + ar_residuals, plot their ACF (lags 1..maxlag) with the +-1.96/sqrt(M)
#   white-noise band (M = number of residuals), and report the FRACTION of lags inside the band.
maxlag = 25
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: are the residuals ~white (most lags inside the band)? If a lag pokes far out, the AR
#   model is still leaving structure in the data -> do NOT trust its spectrum yet.


## 4. Live sanity check

Tie it together with assertions that must hold for an honest fit: the data-driven order is **small** (≤ 4, matching the single pole pair), the residuals are **≈ white** (most ACF lags inside the band), and the matched AR(2) peak sits at the true resonance.

In [ ]:
# --- live sanity check: an honest fit is LOW-ORDER and leaves WHITE residuals (numbers computed live) ---
a2c, s2c = yule_walker(x, 2)
f2, S2 = ar_psd(a2c, s2c)
f_peak2 = f2[np.argmax(S2)]
assert sel_order <= 4, f"selected order {sel_order} is not small — suspiciously high (over-fit risk)"
assert frac_inside >= 0.8, f"only {frac_inside:.0%} of residual-ACF lags inside band — residuals not white"
assert abs(f_peak2 - f0) < 0.02, f"AR(2) peak {f_peak2:.3f} not at the true resonance {f0}"
print(f"sanity PASSED: selected order = {sel_order} (<=4);  "
      f"residual ACF {frac_inside:.0%} inside band;  AR(2) peak {f_peak2:.3f} ~ f0={f0}")


## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not getting a sharp-looking spectrum.

1. **Order vs truth.** The BIC minimum and the physics (one pole pair ⇒ order 2) agree here. Which conclusion about the spectrum **stayed stable** as you changed order (the *location* of the main peak) and which **changed** (its sharpness, and the number of peaks)? Quote the peak frequencies you printed at AR(2), AR(8), AR(30).
2. **Whiteness as evidence.** At the selected order the residual ACF is ~white. What would a *non-white* residual ACF (a lag poking well outside the band) tell you — and why does that veto trusting the AR spectrum even if it looks clean?
3. **Transfer.** On a new recording the BIC minimum jumps to order 20 and a second peak appears. Which do you believe — the criterion or your prior that it's AR(2) — and what one check settles it?

**Rule out (name the wrong move).** Give one concrete *wrong* way to run this — e.g. **cranking the AR order to 30** so the spectrum sprouts spurious peaks and reporting one as a finding, **or** trusting the AR spectrum **without ever checking residual whiteness**. Name the requirement it breaks: it violates the **§1.8** demand that the analysis *preserve an honest link between model and data* — a fit you can defend (right order, white residuals), not a spectrum tuned to look convincing. An over-fit AR spectrum is a hallucinated peak; an unchecked one is an unfalsified claim.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic AR(2); illustrative numbers. The lesson is the method: fit, select the order honestly, and let the residuals — not the picture — decide whether you can trust the spectrum.*